# Stage 1: Exploratory Data Analysis, Missing Values Imputation, Visualization & Distribution Normalization
**Project:** 10-Year Coronary Heart Disease (CHD) Risk Prediction (Framingham Heart Study)  
**Author:** Saif Asaad Naeem | Al-Maaqal University — Department of AI Engineering

---
### Notebook Roadmap:
1. **Environment Setup & Data Loading:** Load the raw Framingham dataset (`4,238` patients, `16` clinical attributes).
2. **Missingness Mechanism (MAR) & Clinical Rules Audit:** Inspect missing data profiles and clinical dependencies (`cigsPerDay` vs. `currentSmoker`, `BPMeds` vs. `prevalentHyp`, `glucose` vs. `diabetes`).
3. **Five-Algorithm Imputation Benchmark:** Compare `SimpleImputer (Mean/Mode)`, `SimpleImputer (Median/Mode)`, `KNNImputer (k=5)`, `MICE (BayesianRidge)`, and `Hybrid MissForest (RandomForest + Domain Rules)`.
4. **Presentation-Grade Clinical Visualizations:** Generate high-resolution diagnostic plots for missingness, class imbalance (`84.8%` vs. `15.2%`), relative risk, age progression, and multicollinearity.
5. **Distribution Analysis & Five Power Transformations:** Analyze skewness, kurtosis, and IQR outliers, and benchmark `Sqrt`, `Log1p`, `Box-Cox`, `Yeo-Johnson`, and `QuantileTransformer`.

### 1. Environment Setup and Dataset Loading
Import core scientific computing, statistical, imputation, and visualization libraries, and load `framingham.csv`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis, normaltest, ks_2samp

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import BayesianRidge, LogisticRegression
from sklearn.preprocessing import StandardScaler, PowerTransformer, QuantileTransformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import mean_squared_error, accuracy_score

# Configure publication-grade plot styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10

# Load the raw Framingham dataset
df = pd.read_csv('framingham.csv')
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Total Incomplete Rows: {df.isnull().any(axis=1).sum()} ({df.isnull().any(axis=1).mean()*100:.2f}%)")

### 2. Missing Values Audit & Clinical Missing-At-Random (MAR) Dependencies
Examine the exact percentage of missing records across all features and verify key physiological rules before imputation.

In [ ]:
# Summarize missing counts, percentages, data types, and cardinality
missing_summary = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percent (%)': (df.isnull().sum() / len(df) * 100).round(2),
    'Dtype': df.dtypes,
    'Unique_Values': df.nunique()
})
missing_summary = missing_summary[missing_summary['Missing_Count'] > 0].sort_values(by='Missing_Count', ascending=False)
print("=== Missing Values Summary Table ===")
print(missing_summary)

print("\n1. Active smoking status for patients with missing 'cigsPerDay':")
print(df[df['cigsPerDay'].isnull()]['currentSmoker'].value_counts())

print("\n2. Cross-tabulation of 'BPMeds' vs. chronic hypertension ('prevalentHyp'):")
print(pd.crosstab(df['prevalentHyp'], df['BPMeds'], dropna=False))

print("\n3. Fasting blood 'glucose' distribution stratified by 'diabetes' status:")
print(df.groupby('diabetes')['glucose'].describe()[['count', 'mean', '50%', 'std', 'max']])

### 3. Five-Algorithm Missing Values Imputation Benchmark
Evaluate five distinct imputation algorithms using a masked reconstruction benchmark (measuring continuous Normalized RMSE and categorical accuracy) and 5-Fold Stratified Cross-Validation.

In [ ]:
cont_missing = ['glucose', 'totChol', 'cigsPerDay', 'BMI', 'heartRate']
cat_missing = ['education', 'BPMeds']
cols_with_missing = cont_missing + cat_missing

# Create a masked validation benchmark on complete cases to measure true reconstruction error
complete_df = df.dropna().copy()
X_comp = complete_df.drop('TenYearCHD', axis=1)
missing_rates = (df.drop('TenYearCHD', axis=1).isnull().sum() / len(df)).to_dict()

np.random.seed(42)
masks = {}
X_masked = X_comp.copy()
for col in cols_with_missing:
    if col == 'cigsPerDay':
        smoker_idx = X_comp[X_comp['currentSmoker'] == 1].index
        chosen = np.random.choice(smoker_idx, 35, replace=False)
        mask = X_comp.index.isin(chosen)
    else:
        mask = np.random.rand(len(X_comp)) < missing_rates[col]
        if mask.sum() < 25:
            mask = np.zeros(len(X_comp), dtype=bool)
            mask[np.random.choice(len(X_comp), 25, replace=False)] = True
    masks[col] = mask
    X_masked.loc[X_masked.index[mask], col] = np.nan

imputed_masked = {}

# Algorithm 1: SimpleImputer (Mean / Mode)
d_mean = X_masked.copy()
for c in cont_missing: d_mean[c] = d_mean[c].fillna(d_mean[c].mean())
for c in cat_missing: d_mean[c] = d_mean[c].fillna(d_mean[c].mode()[0])
imputed_masked['1. SimpleImputer (Mean/Mode)'] = d_mean

# Algorithm 2: SimpleImputer (Median / Mode)
d_med = X_masked.copy()
for c in cont_missing: d_med[c] = d_med[c].fillna(d_med[c].median())
for c in cat_missing: d_med[c] = d_med[c].fillna(d_med[c].mode()[0])
imputed_masked['2. SimpleImputer (Median/Mode)'] = d_med

# Algorithm 3: KNNImputer (k=5, Distance-Weighted + Scaled)
scaler_knn = StandardScaler()
knn_imp = KNNImputer(n_neighbors=5, weights='distance')
d_knn = pd.DataFrame(
    scaler_knn.inverse_transform(knn_imp.fit_transform(scaler_knn.fit_transform(X_masked))),
    columns=X_masked.columns, index=X_masked.index
)
d_knn['education'] = d_knn['education'].round().clip(1, 4)
d_knn['BPMeds'] = d_knn['BPMeds'].round().clip(0, 1)
imputed_masked['3. KNNImputer (k=5, Scaled)'] = d_knn

# Algorithm 4: MICE IterativeImputer (BayesianRidge)
mice_imp = IterativeImputer(estimator=BayesianRidge(), max_iter=15, random_state=42)
d_mice = pd.DataFrame(mice_imp.fit_transform(X_masked), columns=X_masked.columns, index=X_masked.index)
d_mice['education'] = d_mice['education'].round().clip(1, 4)
d_mice['BPMeds'] = d_mice['BPMeds'].round().clip(0, 1)
imputed_masked['4. MICE (BayesianRidge)'] = d_mice

# Algorithm 5: Hybrid MissForest (RandomForest IterativeImputer + Domain-Aware Categorical Rules)
rf_est = RandomForestRegressor(n_estimators=20, max_depth=8, random_state=42)
mf_imp = IterativeImputer(estimator=rf_est, max_iter=5, random_state=42)
d_mf = pd.DataFrame(mf_imp.fit_transform(X_masked), columns=X_masked.columns, index=X_masked.index)
d_mf['education'] = X_masked['education'].fillna(X_masked['education'].mode()[0])
d_mf['BPMeds'] = X_masked['BPMeds'].copy()
d_mf.loc[(d_mf['BPMeds'].isnull()) & (d_mf['prevalentHyp'] == 0), 'BPMeds'] = 0.0
d_mf['BPMeds'] = d_mf['BPMeds'].fillna(0.0)
imputed_masked['5. Hybrid MissForest (RF + Domain)'] = d_mf

### 4. Evaluate Imputation Benchmark & Save Winning Dataset
Apply all five imputers to the full dataset, compute 5-Fold CV ROC-AUC scores, and export the winning `Hybrid MissForest` dataset to `framingham_imputed.csv`.

In [ ]:
X_full = df.drop('TenYearCHD', axis=1)
y_full = df['TenYearCHD']
full_imputed_dict = {}

f1 = X_full.copy()
for c in cont_missing: f1[c] = f1[c].fillna(f1[c].mean())
for c in cat_missing: f1[c] = f1[c].fillna(f1[c].mode()[0])
full_imputed_dict['1. SimpleImputer (Mean/Mode)'] = f1

f2 = X_full.copy()
for c in cont_missing: f2[c] = f2[c].fillna(f2[c].median())
for c in cat_missing: f2[c] = f2[c].fillna(f2[c].mode()[0])
full_imputed_dict['2. SimpleImputer (Median/Mode)'] = f2

f3 = pd.DataFrame(scaler_knn.inverse_transform(knn_imp.fit_transform(scaler_knn.fit_transform(X_full))), columns=X_full.columns)
f3['education'] = f3['education'].round().clip(1, 4)
f3['BPMeds'] = f3['BPMeds'].round().clip(0, 1)
full_imputed_dict['3. KNNImputer (k=5, Scaled)'] = f3

f4 = pd.DataFrame(mice_imp.fit_transform(X_full), columns=X_full.columns)
f4['education'] = f4['education'].round().clip(1, 4)
f4['BPMeds'] = f4['BPMeds'].round().clip(0, 1)
full_imputed_dict['4. MICE (BayesianRidge)'] = f4

f5 = pd.DataFrame(mf_imp.fit_transform(X_full), columns=X_full.columns)
f5['education'] = X_full['education'].fillna(X_full['education'].mode()[0])
f5['BPMeds'] = X_full['BPMeds'].copy()
f5.loc[(f5['BPMeds'].isnull()) & (f5['prevalentHyp'] == 0), 'BPMeds'] = 0.0
f5['BPMeds'] = f5['BPMeds'].fillna(0.0)
full_imputed_dict['5. Hybrid MissForest (RF + Domain)'] = f5

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_eval = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=42)
lr_eval = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', random_state=42))

benchmark_rows = []
for name in imputed_masked.keys():
    df_m = imputed_masked[name]
    nrmses = [np.sqrt(mean_squared_error(X_comp.loc[X_comp.index[masks[c]], c], df_m.loc[df_m.index[masks[c]], c])) / X_comp[c].std() for c in cont_missing]
    cat_accs = [accuracy_score(X_comp.loc[X_comp.index[masks[c]], c], df_m.loc[df_m.index[masks[c]], c]) for c in cat_missing]
    gluc_rmse = np.sqrt(mean_squared_error(X_comp.loc[X_comp.index[masks['glucose']], 'glucose'], df_m.loc[df_m.index[masks['glucose']], 'glucose']))
    cigs_rmse = np.sqrt(mean_squared_error(X_comp.loc[X_comp.index[masks['cigsPerDay']], 'cigsPerDay'], df_m.loc[df_m.index[masks['cigsPerDay']], 'cigsPerDay']))
    
    auc_rf = cross_val_score(rf_eval, full_imputed_dict[name], y_full, cv=skf, scoring='roc_auc').mean()
    auc_lr = cross_val_score(lr_eval, full_imputed_dict[name], y_full, cv=skf, scoring='roc_auc').mean()
    
    benchmark_rows.append({
        'Algorithm': name,
        'Avg_NRMSE': round(np.mean(nrmses), 4),
        'Glucose_RMSE': round(gluc_rmse, 2),
        'CigsPerDay_RMSE': round(cigs_rmse, 2),
        'Cat_Accuracy (%)': round(np.mean(cat_accs)*100, 2),
        '5Fold_Avg_ROC_AUC': round((auc_rf + auc_lr)/2, 4)
    })

imp_bench_df = pd.DataFrame(benchmark_rows)
print("\n=== Missing Value Imputation Algorithms Benchmark Table ===")
print(imp_bench_df.to_string(index=False))

# Export the dataset imputed by the winning algorithm (Hybrid MissForest)
df_clean = f5.copy()
df_clean['TenYearCHD'] = y_full.values
df_clean.to_csv('framingham_imputed.csv', index=False)
print("\n[Successfully saved imputed dataset to: framingham_imputed.csv]")

### 5. Exploratory Data Visualizations (Figures 1 to 4)
Generate presentation-ready diagnostic charts illustrating missingness, class imbalance, relative risk, age-decade progression, and feature multicollinearity.

In [ ]:
# Figure 1: Missing Values Profile & Imputation Benchmark Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
missing_s = (df.isnull().sum() / len(df) * 100)
missing_s = missing_s[missing_s > 0].sort_values(ascending=True)
colors_m = ['#64748b' if v < 2 else '#f59e0b' if v < 5 else '#ef4444' for v in missing_s.values]
axes[0].barh(missing_s.index, missing_s.values, color=colors_m, edgecolor='black', alpha=0.88)
for i, v in enumerate(missing_s.values):
    axes[0].text(v + 0.15, i, f"{v:.2f}% ({int(df[missing_s.index[i]].isnull().sum())})", va='center', fontweight='bold')
axes[0].set_xlim(0, 11)
axes[0].set_title('A. Missing Values Profile (Total Incomplete Rows = 13.73%)', fontweight='bold')
axes[0].set_xlabel('Missing Percentage (%)')

short_names = ['Mean/Mode', 'Median/Mode', 'KNN (k=5)', 'MICE (Bayes)', 'MissForest (Best)']
nrmse_vals = imp_bench_df['Avg_NRMSE'].values
bars = axes[1].bar(short_names, nrmse_vals, width=0.5, color=['#94a3b8', '#ef4444', '#3b82f6', '#6366f1', '#10b981'], edgecolor='black')
for bar, v in zip(bars, nrmse_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 0.02, f"{v:.4f}", ha='center', fontweight='bold')
axes[1].set_ylim(0, 1.25)
axes[1].set_title('B. Imputation Algorithms Benchmark: Continuous NRMSE (Lower is Better)', fontweight='bold')
axes[1].set_ylabel('Average Normalized RMSE')
plt.tight_layout()
plt.savefig('fig1_missing_imputation_benchmark.png', bbox_inches='tight')
plt.show()

In [ ]:
# Figure 2: Target Class Imbalance & Clinical Relative Risk
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
vc = df_clean['TenYearCHD'].value_counts()
bars0 = axes[0].bar(['Healthy (Class 0)', '10-Year CHD (Class 1)'], vc.values, color=['#0ea5e9', '#e11d48'], width=0.45, edgecolor='black')
for bar, v in zip(bars0, vc.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 80, f"{v} ({v/len(df_clean)*100:.2f}%)", ha='center', fontweight='bold', fontsize=11)
axes[0].set_ylim(0, 4100)
axes[0].set_title('A. Target Class Imbalance (Imbalance Ratio = 5.58 : 1)', fontweight='bold')
axes[0].set_ylabel('Number of Patients')

risk_factors = ['prevalentStroke', 'diabetes', 'BPMeds', 'prevalentHyp', 'male', 'currentSmoker']
r_no = [df_clean[df_clean[c]==0]['TenYearCHD'].mean()*100 for c in risk_factors]
r_yes = [df_clean[df_clean[c]==1]['TenYearCHD'].mean()*100 for c in risk_factors]
y_pos = np.arange(len(risk_factors))
axes[1].barh(y_pos - 0.18, r_no, height=0.35, label='Condition Absent (0)', color='#94a3b8', edgecolor='black')
axes[1].barh(y_pos + 0.18, r_yes, height=0.35, label='Condition Present (1)', color='#e11d48', edgecolor='black')
for i, (v0, v1) in enumerate(zip(r_no, r_yes)):
    axes[1].text(v1 + 0.6, i + 0.18, f"{v1:.1f}% ({v1/v0:.2f}x Risk)", va='center', fontweight='bold', color='#be123c')
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(risk_factors, fontweight='bold')
axes[1].set_xlim(0, 54)
axes[1].set_xlabel('10-Year CHD Incidence Rate (%)')
axes[1].set_title('B. Clinical Relative Risk by Medical Condition', fontweight='bold')
axes[1].legend(loc='lower right')
plt.tight_layout()
plt.savefig('fig2_imbalance_and_clinical_risk.png', bbox_inches='tight')
plt.show()

In [ ]:
# Figure 3: Age Decade Progression & Systolic Blood Pressure KDE Separation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_clean['Age_Group'] = pd.cut(df_clean['age'], bins=[30, 40, 50, 60, 72], labels=['32-40 yrs', '41-50 yrs', '51-60 yrs', '61-70 yrs'])
age_hyp = df_clean.groupby(['Age_Group', 'prevalentHyp'], observed=False)['TenYearCHD'].mean().unstack() * 100
age_hyp.plot(kind='bar', ax=axes[0], color=['#38bdf8', '#f43f5e'], edgecolor='black', width=0.65)
axes[0].set_title('A. 10-Year CHD Rate by Age Decade & Hypertension Status', fontweight='bold')
axes[0].set_ylabel('CHD Incidence Rate (%)')
axes[0].set_xlabel('Age Decade')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0, fontweight='bold')
axes[0].legend(['Normotensive (0)', 'Hypertensive (1)'], title='Hypertension')
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%.1f%%', padding=3, fontweight='bold', fontsize=9)
axes[0].set_ylim(0, 42)

sns.kdeplot(data=df_clean, x='sysBP', hue='TenYearCHD', common_norm=False, fill=True, palette=['#0ea5e9', '#e11d48'], alpha=0.35, linewidth=2, ax=axes[1])
axes[1].axvline(140, color='#be123c', linestyle='--', linewidth=1.8, label='Clinical Hypertension Threshold (140 mmHg)')
axes[1].set_title('B. Systolic Blood Pressure (sysBP) Density Separation by CHD Status', fontweight='bold')
axes[1].set_xlabel('Systolic Blood Pressure (mmHg)')
axes[1].legend()
plt.tight_layout()
df_clean.drop('Age_Group', axis=1, inplace=True)
plt.savefig('fig3_age_progression_and_interactions.png', bbox_inches='tight')
plt.show()

# Figure 4: Lower-Triangular Correlation Matrix & Multicollinearity Discovery
plt.figure(figsize=(12, 9))
corr = df_clean.corr()
mask_tri = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask_tri, annot=True, fmt='.2f', cmap='coolwarm', vmin=-0.2, vmax=0.85, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix & Multicollinearity Discovery (sysBP-diaBP r=0.78, Smoker-Cigs r=0.77)', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('fig4_correlation_multicollinearity.png', bbox_inches='tight')
plt.show()

### 6. Data Distribution Analysis, IQR Outliers & Five Transformation Algorithms
Compute skewness, kurtosis, and IQR outlier counts across continuous features, and benchmark five mathematical transformations to normalize heavy right-skewed tails without discarding high-risk patients.

In [ ]:
cont_cols = ['age', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']

dist_rows = []
for c in cont_cols:
    s = df_clean[c]
    q25, q75 = np.percentile(s, [25, 75])
    iqr_val = q75 - q25
    outliers_cnt = ((s < (q25 - 1.5 * iqr_val)) | (s > (q75 + 1.5 * iqr_val))).sum()
    dist_rows.append({
        'Feature': c,
        'Mean': round(s.mean(), 2),
        'Median': round(s.median(), 2),
        'Std': round(s.std(), 2),
        'Skewness': round(skew(s), 2),
        'Kurtosis': round(kurtosis(s), 2),
        'IQR': round(iqr_val, 2),
        'Outliers_Count (%)': f"{outliers_cnt} ({outliers_cnt/len(s)*100:.1f}%)"
    })

dist_summary_df = pd.DataFrame(dist_rows).sort_values(by='Skewness', ascending=False)
print("=== Continuous Features Distribution & IQR Outliers Table ===")
print(dist_summary_df.to_string(index=False))

# Benchmark 5 distribution transformation algorithms on the 7 skewed features
skewed_cols = ['glucose', 'sysBP', 'BMI', 'totChol', 'cigsPerDay', 'diaBP', 'heartRate']
X_skew = df_clean[skewed_cols].copy()

X_pos = X_skew.copy()
X_pos['cigsPerDay'] = X_pos['cigsPerDay'] + 1.0  # Box-Cox requires strictly positive values (> 0)

transformations = {
    '0. Original (Untransformed)': X_skew,
    '1. Square Root (np.sqrt)': np.sqrt(X_skew),
    '2. Log1p (np.log1p)': np.log1p(X_skew),
    '3. Box-Cox Transform': pd.DataFrame(PowerTransformer(method='box-cox').fit_transform(X_pos), columns=skewed_cols),
    '4. Yeo-Johnson Transform (Best)': pd.DataFrame(PowerTransformer(method='yeo-johnson').fit_transform(X_skew), columns=skewed_cols),
    '5. Quantile Transform (Normal)': pd.DataFrame(QuantileTransformer(n_quantiles=1000, output_distribution='normal', random_state=42).fit_transform(X_skew), columns=skewed_cols)
}

trans_rows = []
for t_name, t_df in transformations.items():
    trans_rows.append({
        'Transformation_Method': t_name,
        'Mean_|Skewness|': round(np.mean([abs(skew(t_df[c])) for c in skewed_cols]), 4),
        'Mean_|Kurtosis|': round(np.mean([abs(kurtosis(t_df[c])) for c in skewed_cols]), 4),
        'Glucose_Skew': round(skew(t_df['glucose']), 3),
        'sysBP_Skew': round(skew(t_df['sysBP']), 3),
        'BMI_Skew': round(skew(t_df['BMI']), 3)
    })

trans_comp_df = pd.DataFrame(trans_rows)
print("\n=== Distribution Transformation Algorithms Benchmark Table ===")
print(trans_comp_df.to_string(index=False))

In [ ]:
# Figure 5: Before vs. After Yeo-Johnson Transformation on Top 3 Skewed Features
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
top_skewed = ['glucose', 'sysBP', 'BMI']
yj_df = transformations['4. Yeo-Johnson Transform (Best)']

for i, col in enumerate(top_skewed):
    sns.histplot(df_clean[col], kde=True, color='#ef4444', ax=axes[0, i], bins=35)
    axes[0, i].set_title(f"Before: {col} (Skew = {skew(df_clean[col]):.2f})", fontweight='bold')
    
    sns.histplot(yj_df[col], kde=True, color='#10b981', ax=axes[1, i], bins=35)
    axes[1, i].set_title(f"After Yeo-Johnson: {col} (Skew = {skew(yj_df[col]):.2f})", fontweight='bold')

plt.suptitle('Data Distribution Normalization using Winning Yeo-Johnson Power Transformation', fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('fig5_distribution_yeo_johnson_comparison.png', bbox_inches='tight')
plt.show()